# Cuda - pamięć
Jakub Ciszewski

## Zadanie 1 - Transpozycja

### Programy

In [14]:
%%writefile matrix_transpose.cu

#include <stdio.h>
#include <stdlib.h>

constexpr int SIZES[] = {1024, 2048, 4096};
#define BLOCK_SIZE 32
#define NUM_RUNS 20 // Liczba uruchomień do uśrednienia

__global__ void matrix_transpose_naive(int *input, int *output, int size) {
  int indexX = threadIdx.x + blockIdx.x * blockDim.x;
  int indexY = threadIdx.y + blockIdx.y * blockDim.y;
  int index = indexY * size + indexX;
  int transposedIndex = indexX * size + indexY;

  output[transposedIndex] = input[index];
}

__global__ void matrix_transpose_shared(int *input, int *output, int size) {
  __shared__ int sharedMemory [BLOCK_SIZE][BLOCK_SIZE];

  int indexX = threadIdx.x + blockIdx.x * blockDim.x;
  int indexY = threadIdx.y + blockIdx.y * blockDim.y;

  int tindexX = threadIdx.x + blockIdx.y * blockDim.x;
  int tindexY = threadIdx.y + blockIdx.x * blockDim.y;

  int localIndexX = threadIdx.x;
  int localIndexY = threadIdx.y;

  int index = indexY * size + indexX;
  int transposedIndex = tindexY * size + tindexX;

  sharedMemory[localIndexX][localIndexY] = input[index];

  __syncthreads();

  output[transposedIndex] = sharedMemory[localIndexY][localIndexX];
}

void fill_array(int *data, int size) {
  for(int idx = 0; idx < (size * size); idx++)
    data[idx] = idx;
}

int main(void) {
  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  printf("Rozmiar | Kernel Naive (Średni ms) | Kernel Shared (Średni ms) | Przyspieszenie\n");
  printf("-------------------------------------------------------------------------\n");

  for (int s : SIZES) {
    int *a, *b;
    int *d_a, *d_b;

    int size = s * s * sizeof(int);

    a = (int *)malloc(size); fill_array(a, s);
    b = (int *)malloc(size);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 gridSize(s / BLOCK_SIZE, s / BLOCK_SIZE, 1);

    // warm-up
    matrix_transpose_naive<<<gridSize, blockSize>>>(d_a, d_b, s);
    cudaDeviceSynchronize();

    float total_time_naive = 0.0f;
    for (int i = 0; i < NUM_RUNS; i++) {
      float milliseconds = 0;
      cudaEventRecord(start);
      matrix_transpose_naive<<<gridSize, blockSize>>>(d_a, d_b, s);
      cudaEventRecord(stop);
      cudaEventSynchronize(stop);
      cudaEventElapsedTime(&milliseconds, start, stop);
      total_time_naive += milliseconds;
    }
    float avg_time_naive = total_time_naive / NUM_RUNS;


    // warm-up
    matrix_transpose_shared<<<gridSize, blockSize>>>(d_a, d_b, s);
    cudaDeviceSynchronize();

    float total_time_shared = 0.0f;
    for (int i = 0; i < NUM_RUNS; i++) {
      float milliseconds = 0;
      cudaEventRecord(start);
      matrix_transpose_shared<<<gridSize, blockSize>>>(d_a, d_b, s);
      cudaEventRecord(stop);
      cudaEventSynchronize(stop);
      cudaEventElapsedTime(&milliseconds, start, stop);
      total_time_shared += milliseconds;
    }
    float avg_time_shared = total_time_shared / NUM_RUNS;

    // Obliczanie różnicy / przyspieszenia
    float speedup = avg_time_naive / avg_time_shared;

    printf(" %dx%d |         %8.4f ms     |        %8.4f ms      |    %.2fx\n",
           s, s, avg_time_naive, avg_time_shared, speedup);

    // Czyszczenie pamięci dla bieżącego rozmiaru
    free(a); free(b);
    cudaFree(d_a); cudaFree(d_b);
  }

  // Niszczenie obiektów eventów
  cudaEventDestroy(start);
  cudaEventDestroy(stop);

  return 0;
}

Overwriting matrix_transpose.cu


In [15]:
%%shell

nvcc matrix_transpose.cu -o matrix_transpose -arch=sm_75

In [16]:
%%shell
./matrix_transpose

Rozmiar | Kernel Naive (Średni ms) | Kernel Shared (Średni ms) | Przyspieszenie
-------------------------------------------------------------------------
 1024x1024 |           0.1183 ms     |          0.0892 ms      |    1.33x
 2048x2048 |           0.4481 ms     |          0.3332 ms      |    1.34x
 4096x4096 |           1.7502 ms     |          1.2903 ms      |    1.36x


In [35]:
%%writefile matrix_transpose_ncu.cu

#include<stdio.h>
#include<stdlib.h>

#define N 2048
#define BLOCK_SIZE 32

__global__ void matrix_transpose_naive(int *input, int *output) {

	int indexX = threadIdx.x + blockIdx.x * blockDim.x;
	int indexY = threadIdx.y + blockIdx.y * blockDim.y;
	int index = indexY * N + indexX;
	int transposedIndex = indexX * N + indexY;

    // this has discoalesced global memory store
	output[transposedIndex] = input[index];

	// this has discoalesced global memore load
	// output[index] = input[transposedIndex];
}

__global__ void matrix_transpose_shared(int *input, int *output) {

	__shared__ int sharedMemory [BLOCK_SIZE] [BLOCK_SIZE];

	// global index
	int indexX = threadIdx.x + blockIdx.x * blockDim.x;
	int indexY = threadIdx.y + blockIdx.y * blockDim.y;

	// transposed global memory index
	int tindexX = threadIdx.x + blockIdx.y * blockDim.x;
	int tindexY = threadIdx.y + blockIdx.x * blockDim.y;

	// local index
	int localIndexX = threadIdx.x;
	int localIndexY = threadIdx.y;

	int index = indexY * N + indexX;
	int transposedIndex = tindexY * N + tindexX;

	// reading from global memory in coalesed manner and performing tanspose in shared memory
	sharedMemory[localIndexX][localIndexY] = input[index];

	__syncthreads();

	// writing into global memory in coalesed fashion via transposed data in shared memory
	output[transposedIndex] = sharedMemory[localIndexY][localIndexX];
}

//basically just fills the array with index.
void fill_array(int *data) {
	for(int idx=0;idx<(N*N);idx++)
		data[idx] = idx;
}

void print_output(int *a, int *b) {
	printf("\n Original Matrix::\n");
	for(int idx=0;idx<(N*N);idx++) {
		if(idx%N == 0)
			printf("\n");
		printf(" %d ",  a[idx]);
	}
	printf("\n Transposed Matrix::\n");
	for(int idx=0;idx<(N*N);idx++) {
		if(idx%N == 0)
			printf("\n");
		printf(" %d ",  b[idx]);
	}
}
int main(void) {
	int *a, *b;
        int *d_a, *d_b; // device copies of a, b, c

	int size = N * N *sizeof(int);

	// Alloc space for host copies of a, b, c and setup input values
	a = (int *)malloc(size); fill_array(a);
	b = (int *)malloc(size);

	// Alloc space for device copies of a, b, c
	cudaMalloc((void **)&d_a, size);
	cudaMalloc((void **)&d_b, size);

	// Copy inputs to device
	cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
	cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

	dim3 blockSize(BLOCK_SIZE,BLOCK_SIZE,1);
	dim3 gridSize(N/BLOCK_SIZE,N/BLOCK_SIZE,1);

	matrix_transpose_naive<<<gridSize,blockSize>>>(d_a,d_b);

	// Copy result back to host
	// cudaMemcpy(b, d_b, size, cudaMemcpyDeviceToHost);
	// print_output(a,b);

	matrix_transpose_shared<<<gridSize,blockSize>>>(d_a,d_b);

	// Copy result back to host
	cudaMemcpy(b, d_b, size, cudaMemcpyDeviceToHost);
	// print_output(a,b);

	// terminate memories
	free(a);
	free(b);
    cudaFree(d_a);
	cudaFree(d_b);

	return 0;
}

Overwriting matrix_transpose_ncu.cu


In [36]:
%%shell
nvcc matrix_transpose_ncu.cu -o matrix_transpose_ncu -arch=sm_75

In [37]:
%%shell
ncu matrix_transpose_ncu

==PROF== Connected to process 12579 (/content/matrix_transpose_ncu)
==PROF== Profiling "matrix_transpose_naive" - 0: 0%....50%....100% - 9 passes
==PROF== Profiling "matrix_transpose_shared" - 1: 0%....50%....100% - 9 passes
==PROF== Disconnected from process 12579
[12579] matrix_transpose_ncu@127.0.0.1
  matrix_transpose_naive(int *, int *) (64, 64, 1)x(32, 32, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       584.98
    Elapsed Cycles                cycle      432,736
    Memory Throughput                 %        49.25
    DRAM Throughput                   %        38.01
    Duration                         us       739.74
    L1/TEX Cache Throughput           %        98.51
    L2 Cache Through

#### Porównanie metryk sekcji Occupancy (Bez optymalizacji Bank Conflicts)

| Nazwa metryki *(Metric Name)* | Jednostka *(Unit)* | Kernel Naive | Kernel Shared |
| :--- | :---: | :---: | :---: |
| **Theoretical Occupancy** | % | 100.00% | 100.00% |
| **Achieved Occupancy** | % | **57.45%** | **97.26%** |
| **Theoretical Active Warps per SM** | warp | 32.00 | 32.00 |
| **Achieved Active Warps Per SM** | warp | **18.38** | **31.12** |
| **Block Limit SM** | block | 16 | 16 |
| **Block Limit Registers** | block | 4 | 4 |
| **Block Limit Shared Mem** | block | 16 | 8 |
| **Block Limit Warps** | block | 1 | 1 |

In [34]:
%%writefile conflict_solved.cu

#include <stdio.h>
#include <stdlib.h>

constexpr int SIZES[] = {1024, 2048, 4096};
#define BLOCK_SIZE 32
#define NUM_RUNS 20

__global__ void matrix_transpose_naive(int *input, int *output, int n) {
    int indexX = threadIdx.x + blockIdx.x * blockDim.x;
    int indexY = threadIdx.y + blockIdx.y * blockDim.y;
    int index = indexY * n + indexX;
    int transposedIndex = indexX * n + indexY;

    output[transposedIndex] = input[index];
}

__global__ void matrix_transpose_shared(int *input, int *output, int n) {
    __shared__ int sharedMemory [BLOCK_SIZE] [BLOCK_SIZE + 1];

    int indexX = threadIdx.x + blockIdx.x * blockDim.x;
    int indexY = threadIdx.y + blockIdx.y * blockDim.y;

    int tindexX = threadIdx.x + blockIdx.y * blockDim.x;
    int tindexY = threadIdx.y + blockIdx.x * blockDim.y;

    int localIndexX = threadIdx.x;
    int localIndexY = threadIdx.y;

    int index = indexY * n + indexX;
    int transposedIndex = tindexY * n + tindexX;

    sharedMemory[localIndexX][localIndexY] = input[index];

    __syncthreads();

    output[transposedIndex] = sharedMemory[localIndexY][localIndexX];
}

void fill_array(int *data, int n) {
    for(int idx = 0; idx < (n * n); idx++)
        data[idx] = idx;
}

int main(void) {
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    printf("Rozmiar | Kernel Naive (Średni ms) | Kernel Shared (Średni ms) | Przyspieszenie\n");
    printf("-------------------------------------------------------------------------\n");

    for (int n : SIZES) {
        int *a, *b;
        int *d_a, *d_b;

        int size = n * n * sizeof(int);

        a = (int *)malloc(size); fill_array(a, n);
        b = (int *)malloc(size);

        cudaMalloc((void **)&d_a, size);
        cudaMalloc((void **)&d_b, size);

        cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
        cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

        dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE, 1);
        dim3 gridSize(n / BLOCK_SIZE, n / BLOCK_SIZE, 1);

        matrix_transpose_naive<<<gridSize, blockSize>>>(d_a, d_b, n);
        cudaDeviceSynchronize();

        float total_time_naive = 0.0f;
        for (int i = 0; i < NUM_RUNS; i++) {
            float milliseconds = 0;
            cudaEventRecord(start);
            matrix_transpose_naive<<<gridSize, blockSize>>>(d_a, d_b, n);
            cudaEventRecord(stop);
            cudaEventSynchronize(stop);
            cudaEventElapsedTime(&milliseconds, start, stop);
            total_time_naive += milliseconds;
        }
        float avg_time_naive = total_time_naive / NUM_RUNS;

        matrix_transpose_shared<<<gridSize, blockSize>>>(d_a, d_b, n);
        cudaDeviceSynchronize();

        float total_time_shared = 0.0f;
        for (int i = 0; i < NUM_RUNS; i++) {
            float milliseconds = 0;
            cudaEventRecord(start);
            matrix_transpose_shared<<<gridSize, blockSize>>>(d_a, d_b, n);
            cudaEventRecord(stop);
            cudaEventSynchronize(stop);
            cudaEventElapsedTime(&milliseconds, start, stop);
            total_time_shared += milliseconds;
        }
        float avg_time_shared = total_time_shared / NUM_RUNS;

        float speedup = avg_time_naive / avg_time_shared;

        printf(" %dx%d |         %8.4f ms     |        %8.4f ms      |    %.2fx\n",
               n, n, avg_time_naive, avg_time_shared, speedup);

        free(a); free(b);
        cudaFree(d_a); cudaFree(d_b);
    }

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Overwriting conflict_solved.cu


In [18]:
%%shell
nvcc conflict_solved.cu -o conflict_solved -arch=sm_75

In [23]:
%%shell
./conflict_solved

Rozmiar | Kernel Naive (Średni ms) | Kernel Shared (Średni ms) | Przyspieszenie
-------------------------------------------------------------------------
 1024x1024 |           0.1938 ms     |          0.0563 ms      |    3.44x
 2048x2048 |           0.4878 ms     |          0.1675 ms      |    2.91x
 4096x4096 |           1.9144 ms     |          0.6868 ms      |    2.79x


In [41]:
%%writefile conflict_solved_ncu.cu

#include<stdio.h>
#include<stdlib.h>

#define N 2048
#define BLOCK_SIZE 32

__global__ void matrix_transpose_naive(int *input, int *output) {

	int indexX = threadIdx.x + blockIdx.x * blockDim.x;
	int indexY = threadIdx.y + blockIdx.y * blockDim.y;
	int index = indexY * N + indexX;
	int transposedIndex = indexX * N + indexY;

	// this has discoalesced global memory store
	output[transposedIndex] = input[index];

	// this has discoalesced global memore load
	// output[index] = input[transposedIndex];
}

__global__ void matrix_transpose_shared(int *input, int *output) {

	__shared__ int sharedMemory [BLOCK_SIZE] [BLOCK_SIZE + 1];

	// global index
	int indexX = threadIdx.x + blockIdx.x * blockDim.x;
	int indexY = threadIdx.y + blockIdx.y * blockDim.y;

	// transposed global memory index
	int tindexX = threadIdx.x + blockIdx.y * blockDim.x;
	int tindexY = threadIdx.y + blockIdx.x * blockDim.y;

	// local index
	int localIndexX = threadIdx.x;
	int localIndexY = threadIdx.y;

	int index = indexY * N + indexX;
	int transposedIndex = tindexY * N + tindexX;

	// reading from global memory in coalesed manner and performing tanspose in shared memory
	sharedMemory[localIndexX][localIndexY] = input[index];

	__syncthreads();

	// writing into global memory in coalesed fashion via transposed data in shared memory
	output[transposedIndex] = sharedMemory[localIndexY][localIndexX];
}

//basically just fills the array with index.
void fill_array(int *data) {
	for(int idx=0;idx<(N*N);idx++)
		data[idx] = idx;
}

void print_output(int *a, int *b) {
	printf("\n Original Matrix::\n");
	for(int idx=0;idx<(N*N);idx++) {
		if(idx%N == 0)
			printf("\n");
		printf(" %d ",  a[idx]);
	}
	printf("\n Transposed Matrix::\n");
	for(int idx=0;idx<(N*N);idx++) {
		if(idx%N == 0)
			printf("\n");
		printf(" %d ",  b[idx]);
	}
}
int main(void) {
	int *a, *b;
        int *d_a, *d_b; // device copies of a, b, c

	int size = N * N *sizeof(int);

	// Alloc space for host copies of a, b, c and setup input values
	a = (int *)malloc(size); fill_array(a);
	b = (int *)malloc(size);

	// Alloc space for device copies of a, b, c
	cudaMalloc((void **)&d_a, size);
	cudaMalloc((void **)&d_b, size);

	// Copy inputs to device
	cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
	cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

	dim3 blockSize(BLOCK_SIZE,BLOCK_SIZE,1);
	dim3 gridSize(N/BLOCK_SIZE,N/BLOCK_SIZE,1);

	matrix_transpose_naive<<<gridSize,blockSize>>>(d_a,d_b);

	// Copy result back to host
	// cudaMemcpy(b, d_b, size, cudaMemcpyDeviceToHost);
	// print_output(a,b);

	matrix_transpose_shared<<<gridSize,blockSize>>>(d_a,d_b);

	// Copy result back to host
	cudaMemcpy(b, d_b, size, cudaMemcpyDeviceToHost);
	// print_output(a,b);

	// terminate memories
	free(a);
	free(b);
    cudaFree(d_a);
	cudaFree(d_b);

	return 0;
}

Writing conflict_solved_ncu.cu


In [42]:
%%shell

nvcc conflict_solved_ncu.cu -o conflict_solved_ncu -arch=sm_75

In [43]:
%%shell
ncu conflict_solved_ncu

==PROF== Connected to process 13741 (/content/conflict_solved_ncu)
==PROF== Profiling "matrix_transpose_naive" - 0: 0%....50%....100% - 9 passes
==PROF== Profiling "matrix_transpose_shared" - 1: 0%....50%....100% - 9 passes
==PROF== Disconnected from process 13741
[13741] conflict_solved_ncu@127.0.0.1
  matrix_transpose_naive(int *, int *) (64, 64, 1)x(32, 32, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       584.99
    Elapsed Cycles                cycle      431,546
    Memory Throughput                 %        49.25
    DRAM Throughput                   %        38.10
    Duration                         us       737.70
    L1/TEX Cache Throughput           %        98.50
    L2 Cache Throughpu

Porównanie metryk sekcji Occupancy (Po rozwiązaniu Bank Conflicts)

| Nazwa metryki *(Metric Name)* | Jednostka *(Unit)* | Kernel Naive | Kernel Shared *(Conflict Solved)* |
| :--- | :---: | :---: | :---: |
| **Theoretical Occupancy** | % | 100.00% | 100.00% |
| **Achieved Occupancy** | % | **57.48%** | **93.41%** |
| **Theoretical Active Warps per SM** | warp | 32.00 | 32.00 |
| **Achieved Active Warps Per SM** | warp | **18.39** | **29.89** |
| **Block Limit SM** | block | 16 | 16 |
| **Block Limit Registers** | block | 4 | 4 |
| **Block Limit Shared Mem** | block | 16 | 7 |
| **Block Limit Warps** | block | 1 | 1 |

In [26]:
%%writefile a_bit_pointless.cu

#include <stdio.h>
#include <stdlib.h>

constexpr int SIZES[] = {1024, 2048, 4096};
#define BLOCK_SIZE 32
#define NUM_RUNS 20

__global__ void matrix_copy_coalesced(int *input, int *output, int n) {
    int indexX = threadIdx.x + blockIdx.x * blockDim.x;
    int indexY = threadIdx.y + blockIdx.y * blockDim.y;
    int index = indexY * n + indexX;

    output[index] = input[index];
}

__global__ void matrix_transpose_shared(int *input, int *output, int n) {
    __shared__ int sharedMemory [BLOCK_SIZE] [BLOCK_SIZE + 1];

    int indexX = threadIdx.x + blockIdx.x * blockDim.x;
    int indexY = threadIdx.y + blockIdx.y * blockDim.y;

    int tindexX = threadIdx.x + blockIdx.y * blockDim.x;
    int tindexY = threadIdx.y + blockIdx.x * blockDim.y;

    int localIndexX = threadIdx.x;
    int localIndexY = threadIdx.y;

    int index = indexY * n + indexX;
    int transposedIndex = tindexY * n + tindexX;

    sharedMemory[localIndexX][localIndexY] = input[index];

    __syncthreads();

    output[transposedIndex] = sharedMemory[localIndexY][localIndexX];
}

void fill_array(int *data, int n) {
    for(int idx = 0; idx < (n * n); idx++)
        data[idx] = idx;
}

int main(void) {
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    printf("Rozmiar | Kopiowanie Global (Coalesced ms) | Transpozycja Shared (ms) | Stosunek czasów\n");
    printf("----------------------------------------------------------------------------------------\n");

    for (int n : SIZES) {
        int *a, *b;
        int *d_a, *d_b;

        int size = n * n * sizeof(int);

        a = (int *)malloc(size); fill_array(a, n);
        b = (int *)malloc(size);

        cudaMalloc((void **)&d_a, size);
        cudaMalloc((void **)&d_b, size);

        cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
        cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

        dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE, 1);
        dim3 gridSize(n / BLOCK_SIZE, n / BLOCK_SIZE, 1);

        matrix_copy_coalesced<<<gridSize, blockSize>>>(d_a, d_b, n);
        cudaDeviceSynchronize();

        float total_time_copy = 0.0f;
        for (int i = 0; i < NUM_RUNS; i++) {
            float milliseconds = 0;
            cudaEventRecord(start);
            matrix_copy_coalesced<<<gridSize, blockSize>>>(d_a, d_b, n);
            cudaEventRecord(stop);
            cudaEventSynchronize(stop);
            cudaEventElapsedTime(&milliseconds, start, stop);
            total_time_copy += milliseconds;
        }
        float avg_time_copy = total_time_copy / NUM_RUNS;


        matrix_transpose_shared<<<gridSize, blockSize>>>(d_a, d_b, n);
        cudaDeviceSynchronize();

        float total_time_shared = 0.0f;
        for (int i = 0; i < NUM_RUNS; i++) {
            float milliseconds = 0;
            cudaEventRecord(start);
            matrix_transpose_shared<<<gridSize, blockSize>>>(d_a, d_b, n);
            cudaEventRecord(stop);
            cudaEventSynchronize(stop);
            cudaEventElapsedTime(&milliseconds, start, stop);
            total_time_shared += milliseconds;
        }
        float avg_time_shared = total_time_shared / NUM_RUNS;

        float ratio = avg_time_shared / avg_time_copy;

        printf(" %dx%d |            %8.4f ms         |        %8.4f ms      |    %.2fx\n",
               n, n, avg_time_copy, avg_time_shared, ratio);

        free(a); free(b);
        cudaFree(d_a); cudaFree(d_b);
    }

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Overwriting a_bit_pointless.cu


In [27]:
%%shell
nvcc a_bit_pointless.cu -o a_bit_pointless -arch=sm_75

In [28]:
%%shell
./a_bit_pointless

Rozmiar | Kopiowanie Global (Coalesced ms) | Transpozycja Shared (ms) | Stosunek czasów
----------------------------------------------------------------------------------------
 1024x1024 |              0.0438 ms         |          0.0479 ms      |    1.09x
 2048x2048 |              0.1528 ms         |          0.1674 ms      |    1.10x
 4096x4096 |              0.5958 ms         |          0.6858 ms      |    1.15x


In [44]:
%%writefile a_bit_pointless_ncu.cu

#include <stdio.h>
#include <stdlib.h>

// Ustawiamy rozmiar na sztywno dla profilera
#define N 1024
#define BLOCK_SIZE 32

__global__ void matrix_transpose_naive(int *input, int *output) {
    int indexX = threadIdx.x + blockIdx.x * blockDim.x;
    int indexY = threadIdx.y + blockIdx.y * blockDim.y;
    int index = indexY * N + indexX;
    int transposedIndex = indexX * N + indexY;

    // Discoalesced global memory store
    output[transposedIndex] = input[index];
}

__global__ void matrix_transpose_shared(int *input, int *output) {
    // Rozmiar BLOCK_SIZE + 1 zapobiega konfliktom banków pamięci shared
    __shared__ int sharedMemory [BLOCK_SIZE] [BLOCK_SIZE + 1];

    int indexX = threadIdx.x + blockIdx.x * blockDim.x;
    int indexY = threadIdx.y + blockIdx.y * blockDim.y;

    int tindexX = threadIdx.x + blockIdx.y * blockDim.x;
    int tindexY = threadIdx.y + blockIdx.x * blockDim.y;

    int localIndexX = threadIdx.x;
    int localIndexY = threadIdx.y;

    int index = indexY * N + indexX;
    int transposedIndex = tindexY * N + tindexX;

    // Czytanie coalesced z pamięci globalnej do shared
    sharedMemory[localIndexX][localIndexY] = input[index];

    __syncthreads();

    // Pisanie coalesced z shared do pamięci globalnej
    output[transposedIndex] = sharedMemory[localIndexY][localIndexX];
}

void fill_array(int *data) {
    for(int idx = 0; idx < (N * N); idx++)
        data[idx] = idx;
}

int main(void) {
    int *a, *b;
    int *d_a, *d_b;

    int size = N * N * sizeof(int);

    a = (int *)malloc(size); fill_array(a);
    b = (int *)malloc(size);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dim3 blockSize(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 gridSize(N / BLOCK_SIZE, N / BLOCK_SIZE, 1);

    // Wywołanie nr 1: Naive (profiler przechwyci jako pierwsze)
    matrix_transpose_naive<<<gridSize, blockSize>>>(d_a, d_b);
    cudaDeviceSynchronize();

    // Wywołanie nr 2: Shared z paddingiem (profiler przechwyci jako drugie)
    matrix_transpose_shared<<<gridSize, blockSize>>>(d_a, d_b);
    cudaDeviceSynchronize();

    // Kopiujemy tylko raz na koniec, żeby sprawdzić stabilność, brak wypisywania tekstu
    cudaMemcpy(b, d_b, size, cudaMemcpyDeviceToHost);

    // Czyszczenie zasobów
    free(a); free(b);
    cudaFree(d_a); cudaFree(d_b);

    return 0;
}

Writing a_bit_pointless_ncu.cu


In [46]:
%%shell

nvcc a_bit_pointless_ncu.cu -o a_bit_pointless_ncu -arch=sm_75

In [47]:
%%shell

ncu a_bit_pointless_ncu

==PROF== Connected to process 14921 (/content/a_bit_pointless_ncu)
==PROF== Profiling "matrix_transpose_naive" - 0: 0%....50%....100% - 9 passes
==PROF== Profiling "matrix_transpose_shared" - 1: 0%....50%....100% - 9 passes
==PROF== Disconnected from process 14921
[14921] a_bit_pointless_ncu@127.0.0.1
  matrix_transpose_naive(int *, int *) (32, 32, 1)x(32, 32, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
    SM Frequency                    Mhz       584.90
    Elapsed Cycles                cycle      111,484
    Memory Throughput                 %        47.93
    DRAM Throughput                   %        35.53
    Duration                         us       190.59
    L1/TEX Cache Throughput           %        95.85
    L2 Cache Throughpu

Porównanie metryk sekcji Occupancy (Kopiowanie Coalesced vs Transpozycja Shared)

| Nazwa metryki *(Metric Name)* | Jednostka *(Unit)* | Kernel Coalesced *(Kopiowanie)* | Kernel Shared *(Transpozycja)* |
| :--- | :---: | :---: | :---: |
| **Theoretical Occupancy** | % | 100.00% | 100.00% |
| **Achieved Occupancy** | % | **56.73%** | **98.00%** |
| **Theoretical Active Warps per SM** | warp | 32.00 | 32.00 |
| **Achieved Active Warps Per SM** | warp | **18.15** | **31.36** |
| **Block Limit SM** | block | 16 | 16 |
| **Block Limit Registers** | block | 4 | 4 |
| **Block Limit Shared Mem** | block | 16 | 7 |
| **Block Limit Warps** | block | 1 | 1 |

### Wyniki



Zbiorcze podsumowanie testów wydajności (Rozmiary 1024 - 4096)

| Rozmiar | 1. Naive <br>*(Brak koalescencji)* | 2. Shared <br>*(Z bank conflicts)* | 3. Shared + Padding <br>*(Conflict Solved)* | 4. Global Coalesced <br>*(Czyste kopiowanie)* | Zysk całkowity <br>*(Wersja 1 vs 3)* |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **1024 x 1024** | 0.1938 ms | 0.1183 ms | 0.0479 ms | 0.0438 ms | **4.05x** |
| **2048 x 2048** | 0.4878 ms | 0.4481 ms | 0.1674 ms | 0.1528 ms | **2.91x** |
| **4096 x 4096** | 1.9144 ms | 1.7502 ms | 0.6868 ms | 0.5958 ms | **2.79x** |

---

Zbiorcze porównanie sekcji Occupancy ($1024 \times 1024$)

| Nazwa metryki *(Metric Name)* | Jednostka | 1. Kernel Naive <br>*(Brak koalescencji)* | 2. Kernel Shared <br>*(Z bank conflicts)* | 3. Kernel Shared + Padding / Coalesced |
| :--- | :---: | :---: | :---: | :---: |
| **Theoretical Occupancy** | % | 100.00% | 100.00% | 100.00% |
| **Achieved Occupancy** | % | **57.45%** | **97.26%** | **95,71%** |
| **Theoretical Active Warps per SM** | warp | 32.00 | 32.00 | 32.00 |
| **Achieved Active Warps Per SM** | warp | **18.38** | **31.12** | **30,63** |
| **Block Limit SM** | block | 16 | 16 | 16 |
| **Block Limit Registers** | block | 4 | 4 | 4 |
| **Block Limit Shared Mem** | block | 16 | **8** | **7** |
| **Block Limit Warps** | block | 1 | 1 | 1 |

## Zadanie 2 - Redukcja

### Programy

In [48]:
%%writefile reduction_global_kernel.cu

#include <stdio.h>
#include <stdlib.h>

__global__ void
global_reduction_kernel(float *data_out, float *data_in, int stride, int size)
{
    int idx_x = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx_x + stride < size) {
        data_out[idx_x] += data_in[idx_x + stride];
    }
}

void global_reduction(float *d_out, float *d_in, int n_threads, int size)
{
    int n_blocks = (size + n_threads - 1) / n_threads;
    for (int stride = 1; stride < size; stride *= 2) {
        global_reduction_kernel<<<n_blocks, n_threads>>>(d_out, d_in, stride, size);
    }
}


Writing reduction_global_kernel.cu


In [51]:
%%writefile helper_timer.h

/**
 * Copyright 1993-2013 NVIDIA Corporation.  All rights reserved.
 *
 * Please refer to the NVIDIA end user license agreement (EULA) associated
 * with this source code for terms and conditions that govern your use of
 * this software. Any use, reproduction, disclosure, or distribution of
 * this software and related documentation outside the terms of the EULA
 * is strictly prohibited.
 *
 */

// Helper Timing Functions
#ifndef COMMON_HELPER_TIMER_H_
#define COMMON_HELPER_TIMER_H_

#ifndef EXIT_WAIVED
#define EXIT_WAIVED 2
#endif

// includes, system
#include <vector>

// includes, project
#include "exception.h"

// Definition of the StopWatch Interface, this is used if we don't want to use
// the CUT functions But rather in a self contained class interface
class StopWatchInterface {
 public:
  StopWatchInterface() {}
  virtual ~StopWatchInterface() {}

 public:
  //! Start time measurement
  virtual void start() = 0;

  //! Stop time measurement
  virtual void stop() = 0;

  //! Reset time counters to zero
  virtual void reset() = 0;

  //! Time in msec. after start. If the stop watch is still running (i.e. there
  //! was no call to stop()) then the elapsed time is returned, otherwise the
  //! time between the last start() and stop call is returned
  virtual float getTime() = 0;

  //! Mean time to date based on the number of times the stopwatch has been
  //! _stopped_ (ie finished sessions) and the current total time
  virtual float getAverageTime() = 0;
};

//////////////////////////////////////////////////////////////////
// Begin Stopwatch timer class definitions for all OS platforms //
//////////////////////////////////////////////////////////////////
#if defined(WIN32) || defined(_WIN32) || defined(WIN64) || defined(_WIN64)
// includes, system
#define WINDOWS_LEAN_AND_MEAN
#include <windows.h>
#undef min
#undef max

//! Windows specific implementation of StopWatch
class StopWatchWin : public StopWatchInterface {
 public:
  //! Constructor, default
  StopWatchWin()
      : start_time(),
        end_time(),
        diff_time(0.0f),
        total_time(0.0f),
        running(false),
        clock_sessions(0),
        freq(0),
        freq_set(false) {
    if (!freq_set) {
      // helper variable
      LARGE_INTEGER temp;

      // get the tick frequency from the OS
      QueryPerformanceFrequency(reinterpret_cast<LARGE_INTEGER *>(&temp));

      // convert to type in which it is needed
      freq = (static_cast<double>(temp.QuadPart)) / 1000.0;

      // rememeber query
      freq_set = true;
    }
  }

  // Destructor
  ~StopWatchWin() {}

 public:
  //! Start time measurement
  inline void start();

  //! Stop time measurement
  inline void stop();

  //! Reset time counters to zero
  inline void reset();

  //! Time in msec. after start. If the stop watch is still running (i.e. there
  //! was no call to stop()) then the elapsed time is returned, otherwise the
  //! time between the last start() and stop call is returned
  inline float getTime();

  //! Mean time to date based on the number of times the stopwatch has been
  //! _stopped_ (ie finished sessions) and the current total time
  inline float getAverageTime();

 private:
  // member variables

  //! Start of measurement
  LARGE_INTEGER start_time;
  //! End of measurement
  LARGE_INTEGER end_time;

  //! Time difference between the last start and stop
  float diff_time;

  //! TOTAL time difference between starts and stops
  float total_time;

  //! flag if the stop watch is running
  bool running;

  //! Number of times clock has been started
  //! and stopped to allow averaging
  int clock_sessions;

  //! tick frequency
  double freq;

  //! flag if the frequency has been set
  bool freq_set;
};

// functions, inlined

////////////////////////////////////////////////////////////////////////////////
//! Start time measurement
////////////////////////////////////////////////////////////////////////////////
inline void StopWatchWin::start() {
  QueryPerformanceCounter(reinterpret_cast<LARGE_INTEGER *>(&start_time));
  running = true;
}

////////////////////////////////////////////////////////////////////////////////
//! Stop time measurement and increment add to the current diff_time summation
//! variable. Also increment the number of times this clock has been run.
////////////////////////////////////////////////////////////////////////////////
inline void StopWatchWin::stop() {
  QueryPerformanceCounter(reinterpret_cast<LARGE_INTEGER *>(&end_time));
  diff_time = static_cast<float>(((static_cast<double>(end_time.QuadPart) -
                                   static_cast<double>(start_time.QuadPart)) /
                                  freq));

  total_time += diff_time;
  clock_sessions++;
  running = false;
}

////////////////////////////////////////////////////////////////////////////////
//! Reset the timer to 0. Does not change the timer running state but does
//! recapture this point in time as the current start time if it is running.
////////////////////////////////////////////////////////////////////////////////
inline void StopWatchWin::reset() {
  diff_time = 0;
  total_time = 0;
  clock_sessions = 0;

  if (running) {
    QueryPerformanceCounter(reinterpret_cast<LARGE_INTEGER *>(&start_time));
  }
}

////////////////////////////////////////////////////////////////////////////////
//! Time in msec. after start. If the stop watch is still running (i.e. there
//! was no call to stop()) then the elapsed time is returned added to the
//! current diff_time sum, otherwise the current summed time difference alone
//! is returned.
////////////////////////////////////////////////////////////////////////////////
inline float StopWatchWin::getTime() {
  // Return the TOTAL time to date
  float retval = total_time;

  if (running) {
    LARGE_INTEGER temp;
    QueryPerformanceCounter(reinterpret_cast<LARGE_INTEGER *>(&temp));
    retval += static_cast<float>(((static_cast<double>(temp.QuadPart) -
                                   static_cast<double>(start_time.QuadPart)) /
                                  freq));
  }

  return retval;
}

////////////////////////////////////////////////////////////////////////////////
//! Time in msec. for a single run based on the total number of COMPLETED runs
//! and the total time.
////////////////////////////////////////////////////////////////////////////////
inline float StopWatchWin::getAverageTime() {
  return (clock_sessions > 0) ? (total_time / clock_sessions) : 0.0f;
}
#else
// Declarations for Stopwatch on Linux and Mac OSX
// includes, system
#include <sys/time.h>
#include <ctime>

//! Windows specific implementation of StopWatch
class StopWatchLinux : public StopWatchInterface {
 public:
  //! Constructor, default
  StopWatchLinux()
      : start_time(),
        diff_time(0.0),
        total_time(0.0),
        running(false),
        clock_sessions(0) {}

  // Destructor
  virtual ~StopWatchLinux() {}

 public:
  //! Start time measurement
  inline void start();

  //! Stop time measurement
  inline void stop();

  //! Reset time counters to zero
  inline void reset();

  //! Time in msec. after start. If the stop watch is still running (i.e. there
  //! was no call to stop()) then the elapsed time is returned, otherwise the
  //! time between the last start() and stop call is returned
  inline float getTime();

  //! Mean time to date based on the number of times the stopwatch has been
  //! _stopped_ (ie finished sessions) and the current total time
  inline float getAverageTime();

 private:
  // helper functions

  //! Get difference between start time and current time
  inline float getDiffTime();

 private:
  // member variables

  //! Start of measurement
  struct timeval start_time;

  //! Time difference between the last start and stop
  float diff_time;

  //! TOTAL time difference between starts and stops
  float total_time;

  //! flag if the stop watch is running
  bool running;

  //! Number of times clock has been started
  //! and stopped to allow averaging
  int clock_sessions;
};

// functions, inlined

////////////////////////////////////////////////////////////////////////////////
//! Start time measurement
////////////////////////////////////////////////////////////////////////////////
inline void StopWatchLinux::start() {
  gettimeofday(&start_time, 0);
  running = true;
}

////////////////////////////////////////////////////////////////////////////////
//! Stop time measurement and increment add to the current diff_time summation
//! variable. Also increment the number of times this clock has been run.
////////////////////////////////////////////////////////////////////////////////
inline void StopWatchLinux::stop() {
  diff_time = getDiffTime();
  total_time += diff_time;
  running = false;
  clock_sessions++;
}

////////////////////////////////////////////////////////////////////////////////
//! Reset the timer to 0. Does not change the timer running state but does
//! recapture this point in time as the current start time if it is running.
////////////////////////////////////////////////////////////////////////////////
inline void StopWatchLinux::reset() {
  diff_time = 0;
  total_time = 0;
  clock_sessions = 0;

  if (running) {
    gettimeofday(&start_time, 0);
  }
}

////////////////////////////////////////////////////////////////////////////////
//! Time in msec. after start. If the stop watch is still running (i.e. there
//! was no call to stop()) then the elapsed time is returned added to the
//! current diff_time sum, otherwise the current summed time difference alone
//! is returned.
////////////////////////////////////////////////////////////////////////////////
inline float StopWatchLinux::getTime() {
  // Return the TOTAL time to date
  float retval = total_time;

  if (running) {
    retval += getDiffTime();
  }

  return retval;
}

////////////////////////////////////////////////////////////////////////////////
//! Time in msec. for a single run based on the total number of COMPLETED runs
//! and the total time.
////////////////////////////////////////////////////////////////////////////////
inline float StopWatchLinux::getAverageTime() {
  return (clock_sessions > 0) ? (total_time / clock_sessions) : 0.0f;
}
////////////////////////////////////////////////////////////////////////////////

////////////////////////////////////////////////////////////////////////////////
inline float StopWatchLinux::getDiffTime() {
  struct timeval t_time;
  gettimeofday(&t_time, 0);

  // time difference in milli-seconds
  return static_cast<float>(1000.0 * (t_time.tv_sec - start_time.tv_sec) +
                            (0.001 * (t_time.tv_usec - start_time.tv_usec)));
}
#endif  // WIN32

////////////////////////////////////////////////////////////////////////////////
//! Timer functionality exported

////////////////////////////////////////////////////////////////////////////////
//! Create a new timer
//! @return true if a time has been created, otherwise false
//! @param  name of the new timer, 0 if the creation failed
////////////////////////////////////////////////////////////////////////////////
inline bool sdkCreateTimer(StopWatchInterface **timer_interface) {
// printf("sdkCreateTimer called object %08x\n", (void *)*timer_interface);
#if defined(WIN32) || defined(_WIN32) || defined(WIN64) || defined(_WIN64)
  *timer_interface = reinterpret_cast<StopWatchInterface *>(new StopWatchWin());
#else
  *timer_interface =
      reinterpret_cast<StopWatchInterface *>(new StopWatchLinux());
#endif
  return (*timer_interface != NULL) ? true : false;
}

////////////////////////////////////////////////////////////////////////////////
//! Delete a timer
//! @return true if a time has been deleted, otherwise false
//! @param  name of the timer to delete
////////////////////////////////////////////////////////////////////////////////
inline bool sdkDeleteTimer(StopWatchInterface **timer_interface) {
  // printf("sdkDeleteTimer called object %08x\n", (void *)*timer_interface);
  if (*timer_interface) {
    delete *timer_interface;
    *timer_interface = NULL;
  }

  return true;
}

////////////////////////////////////////////////////////////////////////////////
//! Start the time with name \a name
//! @param name  name of the timer to start
////////////////////////////////////////////////////////////////////////////////
inline bool sdkStartTimer(StopWatchInterface **timer_interface) {
  // printf("sdkStartTimer called object %08x\n", (void *)*timer_interface);
  if (*timer_interface) {
    (*timer_interface)->start();
  }

  return true;
}

////////////////////////////////////////////////////////////////////////////////
//! Stop the time with name \a name. Does not reset.
//! @param name  name of the timer to stop
////////////////////////////////////////////////////////////////////////////////
inline bool sdkStopTimer(StopWatchInterface **timer_interface) {
  // printf("sdkStopTimer called object %08x\n", (void *)*timer_interface);
  if (*timer_interface) {
    (*timer_interface)->stop();
  }

  return true;
}

////////////////////////////////////////////////////////////////////////////////
//! Resets the timer's counter.
//! @param name  name of the timer to reset.
////////////////////////////////////////////////////////////////////////////////
inline bool sdkResetTimer(StopWatchInterface **timer_interface) {
  // printf("sdkResetTimer called object %08x\n", (void *)*timer_interface);
  if (*timer_interface) {
    (*timer_interface)->reset();
  }

  return true;
}

////////////////////////////////////////////////////////////////////////////////
//! Return the average time for timer execution as the total time
//! for the timer dividied by the number of completed (stopped) runs the timer
//! has made.
//! Excludes the current running time if the timer is currently running.
//! @param name  name of the timer to return the time of
////////////////////////////////////////////////////////////////////////////////
inline float sdkGetAverageTimerValue(StopWatchInterface **timer_interface) {
  //  printf("sdkGetAverageTimerValue called object %08x\n", (void
  //  *)*timer_interface);
  if (*timer_interface) {
    return (*timer_interface)->getAverageTime();
  } else {
    return 0.0f;
  }
}

////////////////////////////////////////////////////////////////////////////////
//! Total execution time for the timer over all runs since the last reset
//! or timer creation.
//! @param name  name of the timer to obtain the value of.
////////////////////////////////////////////////////////////////////////////////
inline float sdkGetTimerValue(StopWatchInterface **timer_interface) {
  // printf("sdkGetTimerValue called object %08x\n", (void *)*timer_interface);
  if (*timer_interface) {
    return (*timer_interface)->getTime();
  } else {
    return 0.0f;
  }
}

#endif  // COMMON_HELPER_TIMER_H_



Writing helper_timer.h


In [52]:
%%writefile reduction_global.cpp

#include <stdio.h>
#include <stdlib.h>

// cuda runtime
#include <cuda_runtime.h>
#include <helper_timer.h>

#include "reduction.h"

void run_benchmark(void (*reduce)(float *, float *, int, int),
                   float *d_outPtr, float *d_inPtr, int size);
void init_input(float *data, int size);
float get_cpu_result(float *data, int size);

////////////////////////////////////////////////////////////////////////////////
// Program main
////////////////////////////////////////////////////////////////////////////////
int main(int argc, char *argv[])
{
    float *h_inPtr;
    float *d_inPtr, *d_outPtr;

    unsigned int size = 1 << 24;

    float result_host, result_gpu;

    srand(2019);

    // Allocate memory
    h_inPtr = (float *)malloc(size * sizeof(float));

    // Data initialization with random values
    init_input(h_inPtr, size);

    // Prepare GPU resource
    cudaMalloc((void **)&d_inPtr, size * sizeof(float));
    cudaMalloc((void **)&d_outPtr, size * sizeof(float));

    cudaMemcpy(d_inPtr, h_inPtr, size * sizeof(float), cudaMemcpyHostToDevice);

    // Get reduction result from GPU
    run_benchmark(global_reduction, d_outPtr, d_inPtr, size);
    cudaMemcpy(&result_gpu, &d_outPtr[0], sizeof(float), cudaMemcpyDeviceToHost);

    // Get reduction result from GPU

    // Get all sum from CPU
    result_host = get_cpu_result(h_inPtr, size);
    printf("host: %f, device %f\n", result_host, result_gpu);

    // Terminates memory
    cudaFree(d_outPtr);
    cudaFree(d_inPtr);
    free(h_inPtr);

    return 0;
}

void run_benchmark(void (*reduce)(float *, float *, int, int),
                   float *d_outPtr, float *d_inPtr, int size)
{
    int num_threads = 256;
    int test_iter = 100;

    // warm-up
    reduce(d_outPtr, d_inPtr, num_threads, size);

    // initialize timer
    StopWatchInterface *timer;
    sdkCreateTimer(&timer);
    sdkStartTimer(&timer);

    ////////
    // Operation body
    ////////
    for (int i = 0; i < test_iter; i++)
    {
        cudaMemcpy(d_outPtr, d_inPtr, size * sizeof(float), cudaMemcpyDeviceToDevice);
        reduce(d_outPtr, d_outPtr, num_threads, size);
        //cudaDeviceSynchronize();
    }

    // getting elapsed time
    cudaDeviceSynchronize();
    sdkStopTimer(&timer);

    // Compute and print the performance
    float elapsed_time_msed = sdkGetTimerValue(&timer) / (float)test_iter;
    float bandwidth = size * sizeof(float) / elapsed_time_msed / 1e6;
    printf("Time= %.3f msec, bandwidth= %f GB/s\n", elapsed_time_msed, bandwidth);

    sdkDeleteTimer(&timer);
}

void init_input(float *data, int size)
{
    for (int i = 0; i < size; i++)
    {
        // Keep the numbers small so we don't get truncation error in the sum
        data[i] = (rand() & 0xFF) / (float)RAND_MAX;
    }
}

float get_cpu_result(float *data, int size)
{
    double result = 0.f;
    for (int i = 0; i < size; i++)
        result += data[i];

    return (float)result;
}

Overwriting reduction_global.cpp


In [54]:
%%writefile exception.h
/*
 * Copyright 1993-2017 NVIDIA Corporation.  All rights reserved.
 *
 * Please refer to the NVIDIA end user license agreement (EULA) associated
 * with this source code for terms and conditions that govern your use of
 * this software. Any use, reproduction, disclosure, or distribution of
 * this software and related documentation outside the terms of the EULA
 * is strictly prohibited.
 *
 */

/* CUda UTility Library */
#ifndef COMMON_EXCEPTION_H_
#define COMMON_EXCEPTION_H_

// includes, system
#include <stdlib.h>
#include <exception>
#include <iostream>
#include <stdexcept>
#include <string>

//! Exception wrapper.
//! @param Std_Exception Exception out of namespace std for easy typing.
template <class Std_Exception>
class Exception : public Std_Exception {
 public:
  //! @brief Static construction interface
  //! @return Alwayss throws ( Located_Exception<Exception>)
  //! @param file file in which the Exception occurs
  //! @param line line in which the Exception occurs
  //! @param detailed details on the code fragment causing the Exception
  static void throw_it(const char *file, const int line,
                       const char *detailed = "-");

  //! Static construction interface
  //! @return Alwayss throws ( Located_Exception<Exception>)
  //! @param file file in which the Exception occurs
  //! @param line line in which the Exception occurs
  //! @param detailed details on the code fragment causing the Exception
  static void throw_it(const char *file, const int line,
                       const std::string &detailed);

  //! Destructor
  virtual ~Exception() throw();

 private:
  //! Constructor, default (private)
  Exception();

  //! Constructor, standard
  //! @param str string returned by what()
  explicit Exception(const std::string &str);
};

////////////////////////////////////////////////////////////////////////////////
//! Exception handler function for arbitrary exceptions
//! @param ex exception to handle
////////////////////////////////////////////////////////////////////////////////
template <class Exception_Typ>
inline void handleException(const Exception_Typ &ex) {
  std::cerr << ex.what() << std::endl;

  exit(EXIT_FAILURE);
}

//! Convenience macros

//! Exception caused by dynamic program behavior, e.g. file does not exist
#define RUNTIME_EXCEPTION(msg) \
  Exception<std::runtime_error>::throw_it(__FILE__, __LINE__, msg)

//! Logic exception in program, e.g. an assert failed
#define LOGIC_EXCEPTION(msg) \
  Exception<std::logic_error>::throw_it(__FILE__, __LINE__, msg)

//! Out of range exception
#define RANGE_EXCEPTION(msg) \
  Exception<std::range_error>::throw_it(__FILE__, __LINE__, msg)

////////////////////////////////////////////////////////////////////////////////
//! Implementation

// includes, system
#include <sstream>

////////////////////////////////////////////////////////////////////////////////
//! Static construction interface.
//! @param  Exception causing code fragment (file and line) and detailed infos.
////////////////////////////////////////////////////////////////////////////////
/*static*/ template <class Std_Exception>
void Exception<Std_Exception>::throw_it(const char *file, const int line,
                                        const char *detailed) {
  std::stringstream s;

  // Quiet heavy-weight but exceptions are not for
  // performance / release versions
  s << "Exception in file '" << file << "' in line " << line << "\n"
    << "Detailed description: " << detailed << "\n";

  throw Exception(s.str());
}

////////////////////////////////////////////////////////////////////////////////
//! Static construction interface.
//! @param  Exception causing code fragment (file and line) and detailed infos.
////////////////////////////////////////////////////////////////////////////////
/*static*/ template <class Std_Exception>
void Exception<Std_Exception>::throw_it(const char *file, const int line,
                                        const std::string &msg) {
  throw_it(file, line, msg.c_str());
}

////////////////////////////////////////////////////////////////////////////////
//! Constructor, default (private).
////////////////////////////////////////////////////////////////////////////////
template <class Std_Exception>
Exception<Std_Exception>::Exception() : Std_Exception("Unknown Exception.\n") {}

////////////////////////////////////////////////////////////////////////////////
//! Constructor, standard (private).
//! String returned by what().
////////////////////////////////////////////////////////////////////////////////
template <class Std_Exception>
Exception<Std_Exception>::Exception(const std::string &s) : Std_Exception(s) {}

////////////////////////////////////////////////////////////////////////////////
//! Destructor
////////////////////////////////////////////////////////////////////////////////
template <class Std_Exception>
Exception<Std_Exception>::~Exception() throw() {}

  // functions, exported

#endif  // COMMON_EXCEPTION_H_


Writing exception.h


In [56]:
%%writefile reduction.h
#ifndef _REDUCTION_H_
#define _REDUCTION_H_

// @reduction_kernel.cu
void reduction(float *d_out, float *d_in, int n_threads, int size);

// @naive_reduction_kernel.cu
void global_reduction(float *d_out, float *d_in, int n_threads, int size);
// void atomic_reduction(float *d_out, float *d_in, int n_threads, int size);

#endif // _REDUCTION_H_

Writing reduction.h


In [59]:
%%writefile reduction_shared_kernel.cu

#include <stdio.h>
#include "reduction.h"

/*
    Parallel sum reduction using shared memory
    - takes log(n) steps for n input elements
    - uses n threads
    - only works for power-of-2 arrays
*/

// cuda thread synchronization
__global__ void
reduction_kernel(float* d_out, float* d_in, unsigned int size)
{
    unsigned int idx_x = blockIdx.x * blockDim.x + threadIdx.x;

    extern __shared__ float s_data[];

    s_data[threadIdx.x] = (idx_x < size) ? d_in[idx_x] : 0.f;

    __syncthreads();

    // do reduction
    for (unsigned int stride = 1; stride < blockDim.x; stride *= 2)
    {
        // thread synchronous reduction
        if ( (idx_x % (stride * 2)) == 0 )
            s_data[threadIdx.x] += s_data[threadIdx.x + stride];

        __syncthreads();
    }

    if (threadIdx.x == 0)
        d_out[blockIdx.x] = s_data[0];
}

void reduction(float *d_out, float *d_in, int n_threads, int size)
{
    cudaMemcpy(d_out, d_in, size * sizeof(float), cudaMemcpyDeviceToDevice);
    while(size > 1)
    {
        int n_blocks = (size + n_threads - 1) / n_threads;
        reduction_kernel<<< n_blocks, n_threads, n_threads * sizeof(float), 0 >>>(d_out, d_out, size);
        size = n_blocks;
    }
}

Writing reduction_shared_kernel.cu


In [60]:
%%writefile reduction_shared.cpp

#include <stdio.h>
#include <stdlib.h>

// cuda runtime
#include <cuda_runtime.h>
#include <helper_timer.h>

#include "reduction.h"

void run_benchmark(void (*reduce)(float*, float*, int, int),
                   float *d_outPtr, float *d_inPtr, int size);
void init_input(float* data, int size);
float get_cpu_result(float *data, int size);

////////////////////////////////////////////////////////////////////////////////
// Program main
////////////////////////////////////////////////////////////////////////////////
int main(int argc, char *argv[])
{
    float *h_inPtr;
    float *d_inPtr, *d_outPtr;

    unsigned int size = 1 << 24;

    float result_host, result_gpu;

    srand(2019);

    // Allocate memory
    h_inPtr = (float*)malloc(size * sizeof(float));

    // Data initialization with random values
    init_input(h_inPtr, size);

    // Prepare GPU resource
    cudaMalloc((void**)& d_inPtr, size * sizeof(float));
    cudaMalloc((void**)&d_outPtr, size * sizeof(float));

    cudaMemcpy(d_inPtr, h_inPtr, size * sizeof(float), cudaMemcpyHostToDevice);

    // Get reduction result from GPU
    run_benchmark(reduction, d_outPtr, d_inPtr, size);
    cudaMemcpy(&result_gpu, &d_outPtr[0], sizeof(float), cudaMemcpyDeviceToHost);

    // Get reduction result from GPU

    // Get all sum from CPU
    result_host = get_cpu_result(h_inPtr, size);
    printf("host: %f, device %f\n", result_host, result_gpu);

    // Terminates memory
    cudaFree(d_outPtr);
    cudaFree(d_inPtr);
    free(h_inPtr);

    return 0;
}

void run_benchmark(void (*reduce)(float*, float*, int, int),
              float *d_outPtr, float *d_inPtr, int size)
{
    int num_threads = 256;
    int test_iter = 100;

    // warm-up
    reduce(d_outPtr, d_inPtr, num_threads, size);

    // initialize timer
    StopWatchInterface *timer;
    sdkCreateTimer(&timer);
    sdkStartTimer(&timer);

    ////////
    // Operation body
    ////////
    for (int i = 0; i < test_iter; i++) {
        reduce(d_outPtr, d_inPtr, num_threads, size);
    }

    // getting elapsed time
    cudaDeviceSynchronize();
    sdkStopTimer(&timer);

    // Compute and print the performance
    float elapsed_time_msed = sdkGetTimerValue(&timer) / (float)test_iter;
    float bandwidth = size * sizeof(float) / elapsed_time_msed / 1e6;
    printf("Time= %.3f msec, bandwidth= %f GB/s\n", elapsed_time_msed, bandwidth);

    sdkDeleteTimer(&timer);
}

void init_input(float *data, int size)
{
    for (int i = 0; i < size; i++)
    {
        // Keep the numbers small so we don't get truncation error in the sum
        data[i] = (rand() & 0xFF) / (float)RAND_MAX;
    }
}

float get_cpu_result(float *data, int size)
{
    double result = 0.f;
    for (int i = 0; i < size; i++)
        result += data[i];

    return (float)result;
}


Writing reduction_shared.cpp


In [57]:
%%shell

nvcc -I. reduction_global_kernel.cu reduction_global.cpp -o reduction_global -arch=sm_75

In [61]:
%%shell
nvcc -I. reduction_shared.cpp reduction_shared_kernel.cu -o reduction_shared -arch=sm_75

In [58]:
%%shell

./reduction_global

Time= 14.319 msec, bandwidth= 4.686580 GB/s
host: 0.996007, device 0.996007


In [62]:
%%shell

./reduction_shared

Time= 2.701 msec, bandwidth= 24.842346 GB/s
host: 0.996007, device 0.996007


### Wyniki

| Typ operacji | Czas wykonania *(Time)* | Przepustowość *(Bandwidth)* | Wynik Host vs Device |
| :--- | :---: | :---: | :---: |
| **Reduction Global** | 2.701 ms *(2701 µs)* | **24.84 GB/s** | Zgodny (0.996007) |
| **Reduction Shared** | 14.319 ms *(14319 µs)* | **4.69 GB/s** | Zgodny (0.996007) |